# Airbnb Listing Recommender — Content-Based Filtering

## Table of Contents
1. Setup
2. Load Data
3. Explore Data
4. Feature Engineering
5. Helper Functions
6. Collect Preferences
7. Recommendations
8. Ablation Study

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [2]:
# Reads the cleaned Airbnb dataset
airbnb = pd.read_csv('airbnb_clean.csv')
airbnb.head()

,Unnamed: 0,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,0,1274691077561855573,https://www.airbnb.com/rooms/1274691077561855573,20251101201932,2025-11-02,city scrape,Beautiful room for rent hosted by Svitlana,Cozy Room for Rent in a Two-Bedroom Apartment<...,https://a0.muscache.com/pictures/miso/Hosting-...,658783772,...,5.0,5.0,5.0,5.0,f,1,0,1,0,1.62
1,1,1274722590671904755,https://www.airbnb.com/rooms/1274722590671904755,20251101201932,2025-11-02,city scrape,Exquisite & Charming Studio in Prime Locations,This stylish place to stay is perfect for grou...,https://a0.muscache.com/pictures/hosting/Hosti...,458668555,...,5.0,5.0,4.5,5.0,f,6,0,6,0,0.21
2,2,1274773188072371454,https://www.airbnb.com/rooms/1274773188072371454,20251101201932,2025-11-02,city scrape,"Private 2 Bedroom Unit, Full Kitchen, Washer/D...",This stylish place to stay is perfect for trav...,https://a0.muscache.com/pictures/hosting/Hosti...,22369387,...,NaN,NaN,NaN,NaN,f,3,2,1,0,NaN
3,3,1274820999428562667,https://www.airbnb.com/rooms/1274820999428562667,20251101201932,2025-11-02,city scrape,"Beautiful, Serene West Village Treehouse",Switch off in this serene one-bedroom oasis on...,https://a0.muscache.com/pictures/miso/Hosting-...,505544503,...,NaN,NaN,NaN,NaN,f,1,1,0,0,NaN
4,4,1274826163511555978,https://www.airbnb.com/rooms/1274826163511555978,20251101201932,2025-11-02,city scrape,Renovated 1BR w/ Spa Shower,Newly renovated 1-bedroom oasis in vibrant Dow...,https://a0.muscache.com/pictures/miso/Hosting-...,493151605,...,5.0,5.0,4.0,5.0,f,4,2,2,0,0.10


## 3. Explore Data

In [3]:
# Checks distribution of listings across boroughs and room types
print("Listings by Borough:")
print(airbnb['neighbourhood_group_cleansed'].value_counts())

print("\nListings by Room Type:")
print(airbnb['room_type'].value_counts())

Listings by Borough:
neighbourhood_group_cleansed
Manhattan        16302
Brooklyn         13253
Queens            5312
Bronx             1121
Staten Island      365
Name: count, dtype: int64

Listings by Room Type:
room_type
Entire home/apt    19427
Private room       16306
Hotel room           361
Shared room          259
Name: count, dtype: int64


## 4. Feature Engineering

Listing attributes are converted into a numeric feature matrix using one-hot encoding for categorical fields (borough, neighbourhood, room type), min-max scaling for guest capacity, and binary flags for amenity keywords.

In [4]:
# Amenities to search for in each listing
AMENITY_KEYWORDS = [
    'wifi', 'kitchen', 'washer', 'air conditioning', 'heating',
    'tv', 'elevator', 'dryer', 'refrigerator', 'dishwasher'
]

def build_feature_matrix(df, amenity_keywords=AMENITY_KEYWORDS):
    features = pd.DataFrame(index=df.index)

    # One-hot encodes location and room type columns
    features = pd.concat([features, pd.get_dummies(df['neighbourhood_group_cleansed'], prefix='borough')], axis=1)
    features = pd.concat([features, pd.get_dummies(df['neighbourhood_group_cleansed'], prefix='neighbourhood')], axis=1)
    features = pd.concat([features, pd.get_dummies(df['room_type'], prefix='room')], axis=1)

    # Scales guest capacity to [0, 1] so it stays on the same magnitude as binary features
    scaler = MinMaxScaler()
    features['accommodates'] = scaler.fit_transform(df[['accommodates']].fillna(1))

    # Adds a binary flag for each amenity keyword
    amenity_text = df['amenities'].fillna('').str.lower()
    for kw in amenity_keywords:
        features[f'amenity_{kw.replace(" ", "_")}'] = amenity_text.str.contains(kw, regex=False).astype(int)

    return features.fillna(0).astype(float), scaler


feature_matrix, acc_scaler = build_feature_matrix(airbnb)
print(f"Feature matrix: {feature_matrix.shape[0]} listings x {feature_matrix.shape[1]} features")

Feature matrix: 36353 listings x 25 features


## 5. Helper Functions

In [5]:
def prompt_choice(prompt, valid_options, allow_skip=False):
    # Displays available options and re-prompts on invalid input
    display = sorted(valid_options)
    print(f"\n{prompt}")
    print("Options:", ", ".join(display))
    if allow_skip:
        print("(Press Enter to skip)")
    while True:
        raw = input("Your choice: ").strip()
        if allow_skip and raw == "":
            return None
        # Case-insensitive match against valid options
        match = next((o for o in valid_options if o.lower() == raw.lower()), None)
        if match:
            return match
        print(f"  '{raw}' not recognised. Valid options: {', '.join(display)}")


def prompt_multi_choice(prompt, valid_options):
    # Accepts one or more comma-separated values and re-prompts on any typo
    display = sorted(valid_options)
    print(f"\n{prompt}")
    print("Options:", ", ".join(display))
    print("Enter one or more, comma-separated (e.g. wifi, kitchen). Press Enter to skip.")
    while True:
        raw = input("Your choices: ").strip()
        if raw == "":
            return []
        # Splits input and checks each token against valid options
        tokens = [t.strip() for t in raw.split(",")]
        matched, bad = [], []
        for t in tokens:
            m = next((o for o in valid_options if o.lower() == t.lower()), None)
            (matched if m else bad).append(m or t)
        if bad:
            print(f"  Not recognised: {', '.join(bad)}. Re-enter all choices.")
        else:
            return matched


def prompt_int(prompt, min_val=1, max_val=None):
    # Asks for an integer within the valid range and re-prompts on bad input
    range_hint = f"(min {min_val}" + (f", max {max_val}" if max_val else "") + ")"
    print(f"\n{prompt} {range_hint}")
    print("Press Enter to skip.")
    while True:
        raw = input("Your choice: ").strip()
        if raw == "":
            return None
        try:
            val = int(raw)
            if val < min_val or (max_val and val > max_val):
                print(f"  Please enter a number between {min_val} and {max_val or '∞'}.")
            else:
                return val
        except ValueError:
            print(f"  '{raw}' is not a valid integer.")


def encode_user_preferences(preferences, feature_columns, acc_scaler,
                             amenity_keywords=AMENITY_KEYWORDS):
    # Starts with a zero vector matching the feature matrix shape
    user_vec = pd.Series(0.0, index=feature_columns)

    # Sets the matching column to 1 for each specified preference
    if preferences.get('borough'):
        col = f"borough_{preferences['borough']}"
        if col in user_vec.index:
            user_vec[col] = 1.0

    if preferences.get('room_type'):
        col = f"room_{preferences['room_type']}"
        if col in user_vec.index:
            user_vec[col] = 1.0

    if preferences.get('neighbourhood'):
        col = f"neighbourhood_{preferences['neighbourhood']}"
        if col in user_vec.index:
            user_vec[col] = 1.0

    # Scales the accommodation value to match the feature matrix range
    if preferences.get('min_accommodates'):
        user_vec['accommodates'] = acc_scaler.transform([[preferences['min_accommodates']]])[0][0]

    # Flags each requested amenity
    for kw in preferences.get('amenities_keywords', []):
        col = f"amenity_{kw.lower().replace(' ', '_')}"
        if col in user_vec.index:
            user_vec[col] = 1.0

    return user_vec.values.reshape(1, -1)


def content_based_recommend(airbnb, feature_matrix, user_vec, top_n=10):
    # Computes cosine similarity between the user vector and all listings
    sims = cosine_similarity(user_vec, feature_matrix.values)[0]

    # Keeps only the columns relevant for display
    display_cols = [
        'id', 'name', 'neighbourhood_cleansed', 'neighbourhood_group',
        'room_type', 'property_type', 'accommodates', 'bathrooms_text',
        'host_is_superhost', 'number_of_reviews', 'listing_url', 'price'
    ]
    display_cols = [c for c in display_cols if c in airbnb.columns]

    results = airbnb[display_cols].copy()
    results['similarity_score'] = sims
    results['host_is_superhost'] = results['host_is_superhost'].map({'t': 'Yes', 'f': 'No'})

    # Sorts by similarity score and returns top N
    return (
        results
        .sort_values('similarity_score', ascending=False)
        .head(top_n)
        .set_index('id')
        .assign(similarity_score=lambda df: df['similarity_score'].round(4))
    )


def collect_user_preferences(airbnb):
    print("\n" + "=" * 60)
    print("   Airbnb Listing Recommender — Enter Your Preferences")
    print("=" * 60)
    print("All fields are optional. Press Enter to skip any field.\n")

    # Pulls valid options directly from the dataset
    boroughs   = sorted(airbnb['neighbourhood_group_cleansed'].dropna().unique())
    room_types = sorted(airbnb['room_type'].dropna().unique())
    max_acc    = int(airbnb['accommodates'].dropna().max())
    prefs = {}

    borough = prompt_choice("Which borough?", boroughs, allow_skip=True)
    if borough:
        prefs['borough'] = borough

    # Filters neighbourhoods to the chosen borough if one was selected
    if 'borough' in prefs:
        neigh_pool = airbnb.loc[
            airbnb['neighbourhood_group_cleansed'] == prefs['borough'],
            'neighbourhood_cleansed'
        ].dropna().unique()
    else:
        neigh_pool = airbnb['neighbourhood_cleansed'].dropna().unique()

    neighbourhood = prompt_choice("Which neighbourhood?", sorted(neigh_pool), allow_skip=True)
    if neighbourhood:
        prefs['neighbourhood'] = neighbourhood

    room_type = prompt_choice("What room type?", room_types, allow_skip=True)
    if room_type:
        prefs['room_type'] = room_type

    min_acc = prompt_int("Minimum number of guests to accommodate?", min_val=1, max_val=max_acc)
    if min_acc:
        prefs['min_accommodates'] = min_acc

    amenities = prompt_multi_choice("Which amenities do you need?", AMENITY_KEYWORDS)
    if amenities:
        prefs['amenities_keywords'] = amenities

    return prefs

## 6. Collect Preferences

In [6]:
# Walks through each preference field interactively
user_preferences = collect_user_preferences(airbnb)
print("\nPreferences recorded:", user_preferences)


   Airbnb Listing Recommender — Enter Your Preferences
All fields are optional. Press Enter to skip any field.


Which borough?
Options: Bronx, Brooklyn, Manhattan, Queens, Staten Island
(Press Enter to skip)
Your choice: Bronx

Which neighbourhood?
Options: Allerton, Baychester, Belmont, Bronxdale, Castle Hill, City Island, Claremont Village, Clason Point, Co-op City, Concourse, Concourse Village, Country Club, East Morrisania, Eastchester, Edenwald, Fieldston, Fordham, Highbridge, Hunts Point, Kingsbridge, Longwood, Melrose, Morris Heights, Morris Park, Morrisania, Mott Haven, Mount Eden, Mount Hope, North Riverdale, Norwood, Olinville, Parkchester, Pelham Bay, Pelham Gardens, Port Morris, Riverdale, Schuylerville, Soundview, Spuyten Duyvil, Throgs Neck, Tremont, Unionport, University Heights, Van Nest, Wakefield, West Farms, Westchester Square, Williamsbridge, Woodlawn
(Press Enter to skip)
Your choice: 

What room type?
Options: Entire home/apt, Hotel room, Private room, Shared ro

## 7. Recommendations

In [7]:
# Encodes preferences into a vector and runs the recommender
user_vec = encode_user_preferences(user_preferences, feature_matrix.columns, acc_scaler)
recommendations = content_based_recommend(airbnb, feature_matrix, user_vec, top_n=10)

print(f"Top 10 Recommendations")
display(recommendations)

Top 10 Recommendations


,name,neighbourhood_cleansed,room_type,property_type,accommodates,bathrooms_text,host_is_superhost,number_of_reviews,listing_url,price,similarity_score
id,,,,,,,,,,,
38392995,Rooms available for rent ASAP.,Wakefield,Private room,Private room in home,1,1 private bath,No,0,https://www.airbnb.com/rooms/38392995,NaN,0.5774
49446258,Quiet and close to the LaGuardia airport,Schuylerville,Private room,Private room in rental unit,1,1 shared bath,No,0,https://www.airbnb.com/rooms/49446258,NaN,0.5774
7010332,Belmont manor,Belmont,Private room,Private room in home,1,1 bath,No,0,https://www.airbnb.com/rooms/7010332,NaN,0.5774
3627326,Spain Room,Concourse Village,Private room,Private room in townhouse,2,1 private bath,No,54,https://www.airbnb.com/rooms/3627326,$98.00,0.5769
4967098,"Bronx, 1Bdrm in 3Bdrm Apt.",Morris Park,Private room,Private room in rental unit,2,1 bath,No,2,https://www.airbnb.com/rooms/4967098,NaN,0.5769
16186828,Cleaning,Kingsbridge,Private room,Private room in rental unit,2,1 bath,No,0,https://www.airbnb.com/rooms/16186828,NaN,0.5769
6557289,1 Room in a 2 Bedroom Available,Longwood,Private room,Private room in rental unit,2,1 bath,No,0,https://www.airbnb.com/rooms/6557289,NaN,0.5769
52618107,Hermosa habitación con mucha luz y acondiciona...,Hunts Point,Private room,Private room in rental unit,3,1 shared bath,No,0,https://www.airbnb.com/rooms/52618107,NaN,0.5756
672584509447841012,NYC 1 Bedroom & Convertible Living Room Uptown/Bx,Kingsbridge,Private room,Private room in rental unit,4,1 private bath,No,0,https://www.airbnb.com/rooms/672584509447841012,$87.00,0.5735


In [8]:
recommendations['price'] = pd.to_numeric(
    recommendations['price'].replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

avg = recommendations['price'].dropna().mean()
std = recommendations['price'].dropna().std()


lower_95 = norm.ppf(0.025, loc=avg, scale=np.sqrt(std))
upper_95 = norm.ppf(0.975, loc=avg, scale=np.sqrt(std))

print(f'Based on your preferences you should pay roughly between ${round(lower_95, 2)} and ${round(upper_95,2)}')

Based on your preferences you should pay roughly between $87.03 and $97.97


## 8. Ablation Study

Drops one preference at a time and re-runs the recommender to show how much each field influences the top result.

In [9]:
def run_ablation(airbnb, feature_matrix, acc_scaler, user_preferences):
    results = []
    # Drops one preference at a time and records the top result
    for drop_key in list(user_preferences.keys()) + [None]:
        label   = f"drop '{drop_key}'" if drop_key else 'all preferences'
        reduced = {k: v for k, v in user_preferences.items() if k != drop_key}
        vec     = encode_user_preferences(reduced, feature_matrix.columns, acc_scaler)
        recs    = content_based_recommend(airbnb, feature_matrix, vec, top_n=1)
        top1    = recs.iloc[0]
        results.append({
            'preferences_used': label,
            'top_listing_id'  : recs.index[0],
            'top_listing_name': str(top1['name'])[:45],
            'similarity_score': top1['similarity_score']
        })
    return pd.DataFrame(results)


# Runs ablation and displays summary table
ablation_df = run_ablation(airbnb, feature_matrix, acc_scaler, user_preferences)
print("Ablation Study")
print("=" * 60)
display(ablation_df)

Ablation Study


,preferences_used,top_listing_id,top_listing_name,similarity_score
0,drop 'borough',1274691077561855573,Beautiful room for rent hosted by Svitlana,0.0000
1,all preferences,38392995,Rooms available for rent ASAP.,0.5774
